In [1]:
import ast
import os
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import numpy as np

def process_replicon_data(fraction_data, label = 'pident_90'):
    typ_chr = fraction_data[fraction_data[f'category-{label}'] == 'typical chromosome']
    min_size = min(typ_chr['size'])
    tra_rep = fraction_data[fraction_data[f'category-{label}'] == 'intermediate replicon']
    if tra_rep.empty:
        return min_size * 0.5
    else:
        tra_max = max(tra_rep['size'])
        return (min_size + tra_max)/2

label = 'pident_90'
all_data = []
mob_data = []
amr_data = []
for genus_name in keep_genus:
    #genus_name = 'Escherichia'
    folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
    contig_info = pd.read_csv(f'{folder}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    target_dir = rf"/active-data/analysis_results/chr_pla/genus/IMG_PR_plasmid/{genus_name}"
    contig_data = pd.read_csv(f'{target_dir}/IMGPR_plasmid_data.tsv', sep='\t')
    
    plas_data = pd.read_csv(f'{target_dir}/IMGPR_plasmid_plasmidness.tsv', sep='\t')
    plas_data = plas_data.rename(columns={'contig': 'plasmid_id'})
    merged_data = pd.merge(contig_data, plas_data, on='plasmid_id', how='left')
    merged_data = merged_data[(merged_data['source_type']=='Metagenome')]# & (merged_data['putatively_complete']=='Yes')]

    if merged_data.empty:
        continue

    size_split = process_replicon_data(contig_info, label)
    threshold = 0.3
    conditions = [
        (merged_data['size'] >= size_split) & (merged_data['average plasmid fraction-pident_90'] < threshold),
        (merged_data['size'] < size_split) & (merged_data['average plasmid fraction-pident_90'] < threshold),
        merged_data['average plasmid fraction-pident_90'] >= threshold
    ]
    
    choices = ['typical chromosome', 'intermediate replicon', 'typical plasmid']
    merged_data['category-pident_90'] = np.select(conditions, choices, default='unknown')
    
    mob_table = pd.read_csv(f'{target_dir}/annotations/IMGPR_metagenome_plasmid_mobtyper_results.tsv', sep='\t', index_col=0)
    mob_table['plasmid_id'] = mob_table['id'].str.split('|').str[0]
    mob_data.append(mob_table)
    
    amr_table = pd.read_csv(f'{target_dir}/annotations/IMGPR_metagenome_plasmid_AMRtyper_results.tsv', sep='\t', index_col=0)
    amr_data.append(amr_table)
    
    merged_data['genus']=genus_name
    all_data.append(merged_data)

mob_data = pd.concat(mob_data, ignore_index=True)
amr_data = pd.concat(amr_data, ignore_index=True)
amr_data = amr_data.drop_duplicates(subset=["plasmid_id"], keep="first")
all_metagenome_pla = pd.concat(all_data, ignore_index=True)

all_metagenome_pla = pd.merge(all_metagenome_pla, mob_data[['plasmid_id', 'predicted_mobility']], on='plasmid_id', how='left')
all_metagenome_pla = pd.merge(all_metagenome_pla, amr_data, on='plasmid_id', how='left')

name_dict = {'average plasmid fraction-original': 'plasmidness-original',
             'average plasmid fraction-pident_90': 'plasmidness-pident_90',
             'average plasmid fraction-pident_95': 'plasmidness-pident_95',}

all_metagenome_pla.rename(columns=name_dict, inplace=True)
all_metagenome_pla = all_metagenome_pla[['plasmid_id', 'size', 'source_type', 'ecosystem', 'host_taxonomy', 'genus', 
                                         'putatively_complete', 'topology', 'plasmidness-original', 'plasmidness-pident_90', 'plasmidness-pident_95', 
                                         'category-pident_90', 'predicted_mobility', 'AMR_type']]

target_dir = '/active-data/analysis_results/chr_pla/genus/suptables'
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)
all_metagenome_pla.to_csv('IMG_PR_metagenome_plasmid_plasmidness_data.tsv', sep='\t', index=False)